# Build a Habit-Streak Visualizer — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Build a Habit-Streak Visualizer** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It walks through
the exact same pipeline as the full lesson and the local `examples/habit-streak-visualizer/` scripts, using the
course's bundled sample check-in data so the heatmap looks interesting immediately -- no logging required first.

No API key, no GPU, nothing beyond `pandas` and `matplotlib` -- everything here runs on plain local data.

See the full [lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/habit-streak-visualizer/index.md)
for the step-by-step walkthrough and the reasoning behind each piece.

In [ ]:
!pip install -q pandas matplotlib

## Load the sample check-in log

The log is a flat CSV: one row per `(date, habit, done)` check-in. Real logs get built up one row at a time with a
CLI (see `checkin.py` in the local example); here we load the course's bundled sample data, which already spans
several months for two habits, with real streaks, a slump, and some gaps.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/examples/habit-streak-visualizer/sample_checkins.csv"
df = pd.read_csv(url, parse_dates=["date"])
df["done"] = df["done"].astype(str).str.lower().isin(["y", "yes", "true", "1"])
df = df.drop_duplicates(subset=["date", "habit"], keep="last").sort_values("date").reset_index(drop=True)
df.head()

## Pick a habit and build a dense daily timeline

The log is *sparse* -- only rows for days someone actually logged. Reindexing onto every calendar day in the range,
filling missing days with `False`, turns that sparse log into the dense day-by-day series both the streak math and
the grid layout need.

In [ ]:
HABIT = "Exercise"  # try "Read 10 pages" too

habit_df = df[df["habit"] == HABIT].set_index("date")["done"]
start, end = df["date"].min(), df["date"].max()
daily = habit_df.reindex(pd.date_range(start, end, freq="D"), fill_value=False)
daily.head()

## Compute streaks

A streak is a run of consecutive calendar days logged `True`, with no gap -- an unlogged day counts the same as an
explicit "missed" day, which keeps the definition simple at the cost of punishing forgetting to log at all the same
as actually skipping the habit.

In [ ]:
def compute_streaks(daily: pd.Series) -> dict:
    longest = 0
    current_run = 0
    for i, done in enumerate(daily):
        current_run = current_run + 1 if done else 0
        longest = max(longest, current_run)
        if i == len(daily) - 1:
            streak_ending_at_last_day = current_run
    return {
        "current_streak": streak_ending_at_last_day,
        "longest_streak": longest,
        "total_done": int(daily.sum()),
        "total_days": len(daily),
    }

stats = compute_streaks(daily)
stats

## Lay the days out into a GitHub-style grid

Seven rows (one per weekday) by N columns (one per week), read left-to-right within each column, GitHub-contributions
-graph style. The key trick: the column for each date is `(date - anchor).days // 7`, a plain day-offset from a fixed
Monday anchor -- **not** `date.isocalendar()[1]` (the ISO week number). ISO weeks reset to 1 every January, so a log
spanning a year boundary would have late-December and early-January dates collide into the same low week numbers,
scrambling the grid. An offset from a fixed anchor only ever increases, no matter how many years the log spans.

Cell intensity grows with the *current* streak length that day is part of (capped), so a long run of check-ins
visibly darkens as it builds, instead of every "done" day looking identical.

In [ ]:
import numpy as np

def build_grid(daily: pd.Series):
    dates = daily.index
    anchor = dates[0] - pd.Timedelta(days=dates[0].weekday())  # Monday on/before the first day
    weeks = (dates - anchor).days // 7
    rows = dates.weekday  # 0=Monday .. 6=Sunday

    num_weeks = int(weeks.max()) + 1
    grid = np.full((7, num_weeks), np.nan)

    run, cap = 0, 10
    intensity = []
    for done in daily:
        run = run + 1 if done else 0
        intensity.append(min(run, cap) / cap if done else 0.0)

    for row, week, value in zip(rows, weeks, intensity):
        grid[row, week] = value

    return grid, dates

grid, dates = build_grid(daily)
grid.shape

## Render the heatmap

A single-hue sequential ramp (light -> dark blue): pale means "no streak yet," deep blue means "many days in a row,"
and flat gray marks cells outside the logged date range (before the first day, or after the last, since neither is
guaranteed to land exactly on a Monday/Sunday).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

sequential_blue = LinearSegmentedColormap.from_list(
    "habit_blue", ["#eaf2fc", "#9ec5f4", "#3987e5", "#0d366b"]
)
no_data_gray = "#e8e8ea"

n_rows, n_weeks = grid.shape
fig, ax = plt.subplots(figsize=(max(6, n_weeks * 0.32), 2.4), dpi=120)

display = np.where(np.isnan(grid), 0.0, grid)
ax.imshow(display, cmap=sequential_blue, vmin=0, vmax=1, aspect="equal")

no_data_overlay = np.ma.masked_where(~np.isnan(grid), np.ones_like(grid))
ax.imshow(no_data_overlay, cmap=ListedColormap([no_data_gray]), aspect="equal")

ax.set_yticks(range(n_rows))
ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)

anchor = dates[0] - pd.Timedelta(days=dates[0].weekday())
seen_months, month_starts = set(), []
for date in dates:
    key = (date.year, date.month)
    if key not in seen_months:
        seen_months.add(key)
        month_starts.append(date)
tick_positions = [(d - anchor).days // 7 for d in month_starts]
ax.set_xticks(tick_positions)
ax.set_xticklabels([d.strftime("%b") for d in month_starts], fontsize=8)

ax.set_xticks(np.arange(-0.5, n_weeks, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title(
    f"{HABIT} -- check-in streaks (current: {stats['current_streak']}, longest: {stats['longest_streak']})",
    fontsize=11, loc="left",
)
fig.tight_layout()
plt.show()

## Where to go from here

Swap `HABIT` above for `"Read 10 pages"` and rerun to see a sparser, less consistent history render very
differently. Back in the full local project, `checkin.py` lets you log real check-ins for your own habits day by
day, and `visualize.py` regenerates this exact chart from your own `checkins.csv`.

Be honest with yourself about the tradeoff, though: this is a lower-fidelity way to experience the project than a
real local `uv` project -- no separate files, no real project structure, just cells in a notebook. Treat it as a
quick way to explore, not the primary path.